# Tokenizer Pipeline
# Expects training_data_long.txt to already exist (produced by consolidate_data.ipynb)
# 1. Train or load BPE tokenizer (controlled by TRAIN_NEW_TOKENIZER)
# 2. Encode text to train_long.bin

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/sparkyllm/datasets_pretrain"
INPUT_TEXT = os.path.join(BASE_PATH, "training_data_long.txt")

# Verify consolidate_data.ipynb has been run
if not os.path.exists(INPUT_TEXT):
    raise FileNotFoundError(
        f"{INPUT_TEXT} not found. Run consolidate_data.ipynb first."
    )

size_mb = os.path.getsize(INPUT_TEXT) / 1024 / 1024
print(f"Input: {INPUT_TEXT} ({size_mb:.1f} MB)")

In [ ]:
import json
import time
import random
import numpy as np
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# ================= CONFIGURATION =================
TRAIN_NEW_TOKENIZER = True  # SET TO TRUE TO RETRAIN WITH NEW VOCAB SIZE

INPUT_TEXT = os.path.join(BASE_PATH, "training_data_long.txt")
OUTPUT_DIR = os.path.join(BASE_PATH, "tokenizer_out")
TOKENIZER_PATH = os.path.join(OUTPUT_DIR, "tokenizer.json")

VOCAB_SIZE = 32000
TRAIN_SIZE_BYTES = 200 * 1024 * 1024  # 200 MB subset for vocab training
ENCODE_BATCH_SIZE = 10_000  # lines per batch encode call

EOT_TOKEN = "<|endoftext|>"
UNK_TOKEN = "<|unk|>"

os.makedirs(OUTPUT_DIR, exist_ok=True)

input_size = os.path.getsize(INPUT_TEXT)
input_mb = input_size / 1024 / 1024

# ================= TRAIN OR LOAD TOKENIZER =================
if TRAIN_NEW_TOKENIZER:
    subset_path = os.path.join(OUTPUT_DIR, "bpe_subset.txt")
    print(f"[1/3] Building vocab training subset ({TRAIN_SIZE_BYTES / 1024 / 1024:.0f} MB from {input_mb:.0f} MB)...")
    print(f"  Sampling randomly across file for representative coverage...")

    t0 = time.time()
    with open(INPUT_TEXT, "r", encoding="utf-8", errors="ignore") as f:
        all_lines = f.readlines()
    print(f"  Read {len(all_lines):,} lines in {time.time() - t0:.1f}s")

    random.seed(42)
    random.shuffle(all_lines)

    current_bytes = 0
    with open(subset_path, "w", encoding="utf-8") as out_f:
        for line in all_lines:
            out_f.write(line)
            current_bytes += len(line.encode("utf-8"))
            if current_bytes >= TRAIN_SIZE_BYTES:
                break
    del all_lines
    print(f"  Subset size: {current_bytes / 1024 / 1024:.2f} MB")

    print(f"[2/3] Training Byte-Level BPE tokenizer (vocab={VOCAB_SIZE})...")
    t0 = time.time()
    tokenizer = Tokenizer(BPE(unk_token=UNK_TOKEN, dropout=None))
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
    tokenizer.decoder = ByteLevelDecoder()

    trainer = BpeTrainer(
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=[EOT_TOKEN, UNK_TOKEN],
    )
    tokenizer.train([subset_path], trainer)
    tokenizer.save(TOKENIZER_PATH)
    print(f"  Done in {time.time() - t0:.1f}s -> {TOKENIZER_PATH}")
else:
    print(f"[1/2] Loading existing tokenizer from {TOKENIZER_PATH}...")
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

vocab_size_actual = tokenizer.get_vocab_size()
eot_id = tokenizer.token_to_id(EOT_TOKEN)
unk_id = tokenizer.token_to_id(UNK_TOKEN)

assert eot_id is not None, "EOT token missing"
assert unk_id is not None, "UNK token missing"

print(f"  Vocab size: {vocab_size_actual} | EOT: {eot_id} | UNK: {unk_id}")

# ================= ENCODE TO BIN (BATCH MODE) =================
bin_path = os.path.join(OUTPUT_DIR, "train_long.bin")
meta_path = os.path.join(OUTPUT_DIR, "train_long_meta.json")

step_label = "[3/3]" if TRAIN_NEW_TOKENIZER else "[2/2]"
print(f"\n{step_label} Encoding {input_mb:.1f} MB -> {bin_path} (batch mode, {ENCODE_BATCH_SIZE} lines/batch)")

token_count = 0
doc_count = 0
bytes_read = 0
line_count = 0
t0 = time.time()
last_print = t0

# Collect lines into batches, encode all at once
batch_lines = []    # non-EOT lines to batch encode
batch_markers = []  # list of ("text", line) or ("eot",) to preserve order

with open(bin_path, "wb") as f_bin:
    with open(INPUT_TEXT, "r", encoding="utf-8", errors="ignore") as f_txt:
        for line in f_txt:
            bytes_read += len(line.encode("utf-8"))
            line_count += 1
            stripped = line.strip()
            if not stripped:
                continue

            if stripped == EOT_TOKEN:
                batch_markers.append(("eot",))
                doc_count += 1
            else:
                batch_markers.append(("text", len(batch_lines)))
                batch_lines.append(stripped)

            # Flush batch
            if len(batch_lines) >= ENCODE_BATCH_SIZE:
                # Batch encode all text lines at once (uses multiple CPU threads)
                encoded = tokenizer.encode_batch(batch_lines)

                # Reassemble in order
                buffer = []
                for marker in batch_markers:
                    if marker[0] == "eot":
                        buffer.append(eot_id)
                    else:
                        buffer.extend(encoded[marker[1]].ids)

                arr = np.array(buffer, dtype=np.uint32)
                f_bin.write(arr.tobytes())
                token_count += len(buffer)

                batch_lines.clear()
                batch_markers.clear()

                now = time.time()
                if now - last_print >= 2.0:
                    pct = bytes_read / input_size * 100
                    elapsed = now - t0
                    mb_per_s = (bytes_read / 1024 / 1024) / elapsed
                    eta = (input_size - bytes_read) / (bytes_read / elapsed) if bytes_read > 0 else 0
                    print(f"  {pct:5.1f}% | {token_count:,} tokens | {doc_count:,} docs | {mb_per_s:.1f} MB/s | ETA {eta:.0f}s")
                    last_print = now

    # Flush remainder
    if batch_lines or batch_markers:
        encoded = tokenizer.encode_batch(batch_lines) if batch_lines else []
        buffer = []
        for marker in batch_markers:
            if marker[0] == "eot":
                buffer.append(eot_id)
            else:
                buffer.extend(encoded[marker[1]].ids)
        if buffer:
            arr = np.array(buffer, dtype=np.uint32)
            f_bin.write(arr.tobytes())
            token_count += len(buffer)

elapsed = time.time() - t0

meta = {
    "num_tokens": token_count,
    "num_documents": doc_count,
    "vocab_size": vocab_size_actual,
    "eot_id": eot_id,
    "unk_id": unk_id,
    "tokenizer": "byte-level-bpe",
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

bin_mb = os.path.getsize(bin_path) / 1024 / 1024
print(f"\nDone in {elapsed:.1f}s!")
print(f"  Input:  {input_mb:.1f} MB ({line_count:,} lines)")
print(f"  Output: {bin_mb:.1f} MB ({token_count:,} tokens, {doc_count:,} docs)")
print(f"  Speed:  {input_mb / elapsed:.1f} MB/s | {token_count / elapsed:,.0f} tokens/s")

In [ ]:
# ================= SAVE MANIFEST =================
import hashlib, datetime

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
MANIFEST_DIR = os.path.join(SPARKYLLM, "manifests")
os.makedirs(MANIFEST_DIR, exist_ok=True)

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def file_info(path):
    if not os.path.exists(path):
        return {"path": path, "exists": False}
    return {
        "path": path,
        "size_mb": round(os.path.getsize(path) / 1024 / 1024, 2),
        "sha256": file_hash(path),
    }

tok_info = file_info(TOKENIZER_PATH)
tokenization_id = tok_info["sha256"][:16]

# Load parent consolidation manifest if it exists
parent_hash = None
consol_path = os.path.join(MANIFEST_DIR, "consolidation_latest.json")
if os.path.exists(consol_path):
    with open(consol_path) as f:
        parent_hash = json.load(f).get("sources_hash")

manifest = {
    "stage": "tokenization",
    "created": datetime.datetime.now().isoformat(),
    "tokenization_id": tokenization_id,
    "parent_sources_hash": parent_hash,
    "input": file_info(INPUT_TEXT),
    "tokenizer": tok_info,
    "token_bin": file_info(bin_path),
    "config": {
        "type": "byte-level-bpe",
        "vocab_size_requested": VOCAB_SIZE,
        "vocab_size_actual": vocab_size_actual,
        "trained_new": TRAIN_NEW_TOKENIZER,
        "eot_id": eot_id,
        "unk_id": unk_id,
    },
    "stats": {
        "num_tokens": token_count,
        "num_documents": doc_count,
    },
}

manifest_path = os.path.join(MANIFEST_DIR, "tokenization_latest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\nManifest saved: {manifest_path}")
print(f"  tokenization_id: {tokenization_id}")
print(f"  parent_sources_hash: {parent_hash}")